# Direct-optimisation baseline — no model at all

Take each `h`, run Adam on the QAOA angles until `P(ground)` stops improving, write the result.
That is the whole notebook.

**Why this is the number that matters.** It is the honest ceiling for the ML approach and the
score any model has to justify itself against. Notebook 01's MLP produced angles that were a
*worse* warm start than a tuned constant, so before building a better model it is worth knowing
what the optimiser alone achieves on the exact competition metric.

**It is also a legal submission.** The rules constrain two things: angles must depend on `h`
(they do — each is optimised for its own instance), and inference over `h_test` must finish inside
10 minutes. The timing cell below checks the second explicitly against a 600 s budget.

**But it is not an ML solution**, and shouldn't be the final one. The task exists to *replace* the
classical optimisation loop; a submission that just runs it would score well on accuracy (70 pts)
and poorly on "quality and innovativeness of the ML solution" (20 pts). Treat this as the
benchmark, and as a safety submission if a model never beats it.

Everything runs through the organisers' `QAOA.py` under plain autograd — no custom simulator, so
there is nothing here to verify beyond their own code.

## 0. Setup

In [ ]:
import os, sys, glob, time, math, csv, json
import numpy as np
import torch
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  {torch.cuda.get_device_name(0)}  "
          f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")
else:
    print("  WARNING: no GPU. This will be slow — lower CFG['n_restarts'] and CFG['steps'].")

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)


def find_file(name):
    """Locate an organiser file across Kaggle input, Colab, or a local checkout."""
    for root in ["/kaggle/input", "/kaggle/working", "data/raw", "../data/raw", ".", "..", "/content"]:
        if os.path.isdir(root):
            hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True))
            if hits:
                return hits[0]
    raise FileNotFoundError(
        f"Could not find {name}. Attach the competition files as a Kaggle dataset "
        f"(J.npy, h_train.npy, QAOA.py) or put them in ./data/raw/.")


QAOA_PATH = find_file("QAOA.py")
sys.path.insert(0, os.path.dirname(os.path.abspath(QAOA_PATH)))
from QAOA import QAOA, P as P_DEPTH, N_QUBITS

J = np.load(find_file("J.npy")).astype(np.float64)
h_train = np.load(find_file("h_train.npy")).astype(np.float64)
qaoa = QAOA(torch.tensor(J, dtype=torch.float32), device=DEVICE)
print(f"\nJ {J.shape} | h_train {h_train.shape} | p={P_DEPTH} | n={N_QUBITS}")

## 1. Configuration

Cost is `n_instances * n_restarts * steps` gradient evaluations. The two knobs that matter:

- **`n_restarts`** — the landscape is multi-modal, so this is not optional. §3 plots best-of-k so
  you can see where it saturates.
- **`chunk_rows`** — GPU memory. Autograd stores ~65 intermediate `(rows, 4096)` complex tensors,
  so peak is roughly `rows * 2 MiB`; 2048 rows ≈ 4.4 GiB. Halve it if you hit OOM.

In [ ]:
CFG = dict(
    n_restarts = 32,     # independent Adam runs per instance, best one kept
    steps      = 500,    # Adam steps
    lr         = 0.06,   # cosine-annealed to lr/25
    chunk_rows = 2048,   # (instance x restart) rows held on the GPU at once
)
QUICK = False            # True -> tiny run to check the notebook end to end
if QUICK:
    CFG.update(n_restarts=8, steps=150)

print(json.dumps(CFG, indent=2))

## 2. The optimiser

Batched Adam ascent on `P(ground)`. Every `(instance, restart)` pair is one row of the batch and
is optimised independently — the GPU does all of them at once.

The restart pool is 1/3 **linear ramp** (a discretised adiabatic schedule, which lands in the good
basin far more often than chance) and 2/3 uniform. The mix matters: the ramp family alone is too
narrow to cover the landscape.

In [ ]:
def init_angles(n, rng, ramp_frac=0.34):
    """Restart pool: linear-ramp family + uniform noise."""
    g = rng.uniform(-np.pi/2, np.pi/2, (n, P_DEPTH))
    b = rng.uniform(-np.pi/2, np.pi/2, (n, P_DEPTH))
    n_ramp = int(n * ramp_frac)
    if n_ramp:
        dt = rng.uniform(0.2, 1.4, (n_ramp, 1))
        l = np.arange(P_DEPTH)[None, :]
        g[:n_ramp] = (l + 1) / P_DEPTH * dt
        b[:n_ramp] = (1 - l / P_DEPTH) * dt
    return g, b


def optimize_angles(h, n_restarts, steps, lr, chunk_rows, seed=0, log=True,
                    return_all_restarts=False):
    """Multi-restart Adam ascent on P(ground) for every row of h.

    Returns (gamma, beta, p_ground) for the best restart of each instance. Angles are whatever
    Adam converged to — no canonicalisation, because nothing downstream has to learn them.
    """
    gen = np.random.default_rng(seed)
    n_h = max(1, chunk_rows // n_restarts)
    G, B, V, ALL = [], [], [], []
    t0 = time.time()

    for s in range(0, len(h), n_h):
        hc = h[s:s + n_h]
        hb = torch.tensor(np.repeat(hc, n_restarts, axis=0), dtype=torch.float32, device=DEVICE)
        g0, b0 = init_angles(len(hc) * n_restarts, gen)
        g = torch.tensor(g0, dtype=torch.float32, device=DEVICE, requires_grad=True)
        b = torch.tensor(b0, dtype=torch.float32, device=DEVICE, requires_grad=True)

        opt = torch.optim.Adam([g, b], lr=lr)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps, eta_min=lr / 25)
        for _ in range(steps):
            opt.zero_grad()
            (-qaoa.p_ground(hb, g, b).sum()).backward()
            opt.step()
            sched.step()

        with torch.no_grad():
            v = qaoa.p_ground(hb, g, b).view(len(hc), n_restarts)
        gg = g.detach().view(len(hc), n_restarts, P_DEPTH)
        bb = b.detach().view(len(hc), n_restarts, P_DEPTH)
        rows = torch.arange(len(hc), device=v.device)
        pick = v.argmax(dim=1)

        G.append(gg[rows, pick].cpu().numpy())
        B.append(bb[rows, pick].cpu().numpy())
        V.append(v[rows, pick].cpu().numpy())
        if return_all_restarts:
            ALL.append(v.cpu().numpy())

        if log:
            done = s + len(hc)
            el = time.time() - t0
            print(f"  {done}/{len(h)}  elapsed {el:6.1f}s  eta {el/done*(len(h)-done):6.1f}s  "
                  f"meanP={np.concatenate(V).mean():.4f}", end="\r")
    if log:
        print()

    out = (np.concatenate(G), np.concatenate(B), np.concatenate(V))
    return out + (np.concatenate(ALL),) if return_all_restarts else out


@torch.no_grad()
def score(h, gamma, beta, batch=1024):
    """Mean P(ground) under the organisers' simulator — the competition metric."""
    out = []
    for i in range(0, len(h), batch):
        out.append(qaoa.p_ground(
            torch.tensor(h[i:i+batch], dtype=torch.float32, device=DEVICE),
            torch.tensor(np.asarray(gamma)[i:i+batch], dtype=torch.float32, device=DEVICE),
            torch.tensor(np.asarray(beta)[i:i+batch], dtype=torch.float32, device=DEVICE),
        ).cpu().numpy())
    return np.concatenate(out)

## 3. How many restarts do we actually need?

Run a small probe with all restarts kept, and look at best-of-k. This sets `n_restarts` honestly
instead of by guesswork, and it is the cheapest way to see how multi-modal the landscape is.

In [ ]:
N_PROBE = 32
t0 = time.time()
_, _, v_best, v_all = optimize_angles(h_train[:N_PROBE], CFG["n_restarts"], CFG["steps"],
                                      CFG["lr"], CFG["chunk_rows"], seed=1, log=False,
                                      return_all_restarts=True)
dt = time.time() - t0
rate = N_PROBE / dt
print(f"probe: {N_PROBE} instances x {CFG['n_restarts']} restarts x {CFG['steps']} steps "
      f"in {dt:.1f}s  ({rate:.2f} instances/s)\n")

print(f"best restart   : {v_all.max(1).mean():.4f}")
print(f"median restart : {np.median(v_all, axis=1).mean():.4f}")
print(f"worst restart  : {v_all.min(1).mean():.4f}")
print("\nbest-of-k (mean over probe instances):")
ks = [k for k in [1, 2, 4, 8, 16, 24, 32, 48, 64] if k <= CFG["n_restarts"]]
curve = []
for k in ks:
    boot = np.array([[v_all[i][np.random.default_rng(s).permutation(v_all.shape[1])[:k]].max()
                      for s in range(64)] for i in range(N_PROBE)])
    curve.append(boot.mean())
    print(f"  k={k:2d}: {boot.mean():.4f}   ({boot.mean()/v_all.max(1).mean()*100:5.1f}% of "
          f"best-of-{CFG['n_restarts']})")

plt.figure(figsize=(5.5, 3.2))
plt.plot(ks, curve, "o-")
plt.xlabel("restarts kept (k)"); plt.ylabel("mean P(ground)")
plt.title("returns on restarts"); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

print(f"\nprojected time for 500 instances at this setting: {500/rate:.0f}s "
      f"({'WITHIN' if 500/rate < 600 else 'OVER'} the 600s inference budget)")

## 4. Run it on all of `h_train`

This is the number the main-stage leaderboard reports, so it is directly comparable.

In [ ]:
t0 = time.time()
gamma, beta, v = optimize_angles(h_train, CFG["n_restarts"], CFG["steps"],
                                 CFG["lr"], CFG["chunk_rows"], seed=SEED)
elapsed = time.time() - t0

p = score(h_train, gamma, beta)          # re-scored with the organisers' simulator
# loose tolerance: float32 reductions differ with batch grouping, and a tight assert here
# would throw away the whole run over 1e-6 of noise
assert np.abs(p - v).max() < 1e-4, "optimiser and scorer disagree"

print(f"\n{'='*56}")
print(f"  DIRECT OPTIMISATION BASELINE  (n={len(h_train)} h_train instances)")
print(f"{'='*56}")
print(f"  mean   P(ground) : {p.mean():.5f}   <- the baseline")
print(f"  median P(ground) : {np.median(p):.5f}")
print(f"  min / max        : {p.min():.5f} / {p.max():.5f}")
print(f"  random-angle floor: {1/2**N_QUBITS:.5f}")
print(f"  wall clock       : {elapsed:.0f}s for {len(h_train)} instances")
print(f"{'='*56}")

plt.figure(figsize=(6, 3))
plt.hist(p, bins=50)
plt.axvline(p.mean(), c="r", ls="--", label=f"mean {p.mean():.4f}")
plt.xlabel("P(ground)"); plt.ylabel("count"); plt.legend()
plt.title("direct optimisation, per instance"); plt.tight_layout(); plt.show()

## 5. Inference-budget check

The rules allow 10 minutes to produce angles for all of `h_test` (500 instances). Direct
optimisation *is* the inference here, so the cost above is the cost that counts.

In [ ]:
per_inst = elapsed / len(h_train)
proj_500 = per_inst * 500
print(f"measured        : {per_inst*1000:.1f} ms / instance")
print(f"projected (500) : {proj_500:.0f}s  of a 600s budget  "
      f"({proj_500/600*100:.0f}% used)")

if proj_500 > 600:
    head = 600 / proj_500
    print(f"\nOVER BUDGET. Scale down by ~{head:.2f}x — e.g. "
          f"n_restarts={max(1,int(CFG['n_restarts']*head))} or steps={int(CFG['steps']*head)}.")
    print("The §3 curve shows what that costs in P(ground).")
else:
    print(f"\nWithin budget with {600-proj_500:.0f}s to spare — room for "
          f"~{600/proj_500:.1f}x more restarts or steps if that curve is still climbing.")

## 6. Submission

Uses `h_test.npy` as soon as it is present, otherwise `h_train.npy` so the cell stays runnable.
Angles are optimised for whichever file is loaded — nothing is reused from the run above.

In [ ]:
def write_submission(path, gamma, beta):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id"] + [f"gamma_{i}" for i in range(P_DEPTH)]
                          + [f"beta_{i}" for i in range(P_DEPTH)])
        for i, (gi, bi) in enumerate(zip(gamma, beta)):
            w.writerow([i] + [f"{x:.8f}" for x in gi] + [f"{x:.8f}" for x in bi])


try:
    h_sub, src = np.load(find_file("h_test.npy")).astype(np.float64), "h_test.npy"
    t0 = time.time()
    g_sub, b_sub, _ = optimize_angles(h_sub, CFG["n_restarts"], CFG["steps"],
                                      CFG["lr"], CFG["chunk_rows"], seed=SEED)
    infer_time = time.time() - t0
except FileNotFoundError:
    h_sub, src = h_train, "h_train.npy (h_test not released yet)"
    g_sub, b_sub, infer_time = gamma, beta, elapsed

write_submission("submission.csv", g_sub, b_sub)
p_sub = score(h_sub, g_sub, b_sub)

print(f"wrote submission.csv from {src}")
print(f"  rows           : {len(h_sub)}")
print(f"  mean P(ground) : {p_sub.mean():.5f}")
print(f"  inference time : {infer_time:.0f}s / 600s budget")

np.savez_compressed("direct_opt_angles.npz", h=h_sub, gamma=g_sub, beta=b_sub, p_ground=p_sub)
print("saved direct_opt_angles.npz (reusable as ML labels or as a warm start)")

## 7. What to do with this number

`mean P(ground)` above is the bar. Concretely:

- **Any model must beat it**, or the model is costing you score rather than earning it. Notebook
  01's MLP did not, and this quantifies by how much.
- **It doubles as a safety submission.** If the inference-budget check passed, this is a legal
  submission you can bank while iterating on a model.
- **`direct_opt_angles.npz` is a labelled dataset** — the same angles, at this quality, are what a
  supervised model would be trained on. Worth remembering that notebook 01 showed high label
  quality does not imply learnable labels: these come from scattered basins, so regressing them
  directly reproduces the same failure.

The gap between this number and what a model achieves is the real measure of the ML contribution.
The way to close it is to stop regressing angles and optimise the metric directly — the model
emits angles, `QAOA.py` turns them into `P(ground)`, and that backpropagates into the weights.
Then the multi-modality that broke the regression stops mattering, because no target is involved.